In [1]:
import pyodbc
import pandas as pd
import numpy as np
import tkinter as tk
from tkinter import ttk, messagebox
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import (
    FigureCanvasTkAgg,
    NavigationToolbar2Tk
)
import matplotlib.dates as mdates
MIN_MANDREL_LENGTH = 40.3
MAX_MANDREL_LENGTH = 40.8
MANDREL_LENGTH = 40.8
START_DATE = "2025-01-01 00:00:00"
BOUNDARY_SEARCH_MINUTES = 2
MIN_OD_DROP = 0.05
DROP_MAD_MULTIPLIER = 6.0
STOP_OD_THRESHOLD = 1.0
MIN_STOP_DURATION_SECONDS = 60
INITIAL_INTERRUPTION_MAX_SECONDS = 200
MIN_MANDREL_COUNT_FOR_REMOVAL = 6
CONNECTION_STRING = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=atwpSQL-STP-app;"
    "DATABASE=LinePC7442;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)
query = """
SELECT
    t1.[timestamp],
    LEFT(
        CAST(t1.[Prog_Nr] AS VARCHAR(MAX)),
        255
    ) AS Prog_Nr,
    t1.[ordername],
    t1.[Linie_ist],
    (t1.[Linie_ist] / 60.0) * 2 AS mandrel_m,
    t1.[Auszen_DM_XY_ist] AS OD_real,
    t1.[Extr_ist] AS rpm,
    t1.[Innen_DM_ist] AS ID_real,
    t1.[Massedruck] AS mass_pressure,
    t1.[Mass_ist] AS mass_temp,
    (
        ISNULL(t1.[Wand_X_links], 0)
      + ISNULL(t1.[Wand_X_rechts], 0)
      + ISNULL(t1.[Wand_Y_links], 0)
      + ISNULL(t1.[Wand_Y_rechts], 0)
    ) / 4.0 AS wt_real,
    CASE
        WHEN LEN(tr.[description])
             - LEN(REPLACE(tr.[description], '_', '')) >= 2
        THEN LEFT(
            tr.[description],
            CHARINDEX('_', tr.[description]) - 1
        )
        ELSE tr.[description]
    END AS compound,
    CASE
        WHEN LEN(tr.[description])
             - LEN(REPLACE(tr.[description], '_', '')) >= 2
        THEN SUBSTRING(
            tr.[description],
            CHARINDEX('_', tr.[description]) + 1,
            CHARINDEX(
                '_',
                tr.[description],
                CHARINDEX('_', tr.[description]) + 1
            )
            - CHARINDEX('_', tr.[description])
            - 1
        )
        ELSE NULL
    END AS ID,
    CASE
        WHEN LEN(tr.[description])
             - LEN(REPLACE(tr.[description], '_', '')) >= 2
        THEN SUBSTRING(
            tr.[description],
            CHARINDEX(
                '_',
                tr.[description],
                CHARINDEX('_', tr.[description]) + 1
            ) + 1,
            LEN(tr.[description])
        )
        ELSE NULL
    END AS wth
FROM dbo.DI_Ex2_Istwerte t1
LEFT JOIN dbo.troester_recipe tr
    ON CAST(t1.[Prog_Nr] AS VARCHAR(255))
     = CAST(tr.[recipename] AS VARCHAR(255))
WHERE t1.[timestamp] >= ?
  AND CAST(
        t1.[Prog_Nr]
        AS VARCHAR(255)
      ) = ?
  AND t1.[ordername] = ?
ORDER BY t1.[timestamp]
"""
class MandrelVisualizer:
    def __init__(self, root):
        self.root = root
        self.root.title(
            "Mandrel / OD_real Visualisation"
        )
        self.root.geometry(
            "1500x900"
        )
        control_frame = ttk.Frame(
            self.root,
            padding=10
        )
        control_frame.pack(
            side=tk.TOP,
            fill=tk.X
        )
        ttk.Label(
            control_frame,
            text="Prog_Nr:"
        ).pack(
            side=tk.LEFT,
            padx=(0, 5)
        )
        self.program_var = tk.StringVar()
        self.program_entry = ttk.Entry(
            control_frame,
            textvariable=self.program_var,
            width=25
        )
        self.program_entry.pack(
            side=tk.LEFT,
            padx=(0, 20)
        )
        ttk.Label(
            control_frame,
            text="Ordername:"
        ).pack(
            side=tk.LEFT,
            padx=(0, 5)
        )
        self.order_var = tk.StringVar()
        self.order_entry = ttk.Entry(
            control_frame,
            textvariable=self.order_var,
            width=40
        )
        self.order_entry.pack(
            side=tk.LEFT,
            padx=(0, 20)
        )
        self.load_button = ttk.Button(
            control_frame,
            text="Load & Plot",
            command=self.load_and_plot
        )
        self.load_button.pack(
            side=tk.LEFT,
            padx=5
        )
        ttk.Button(
            control_frame,
            text="Clear",
            command=self.clear_graph
        ).pack(
            side=tk.LEFT,
            padx=5
        )
        self.status_var = tk.StringVar(
            value="Enter Prog_Nr and Ordername"
        )
        ttk.Label(
            control_frame,
            textvariable=self.status_var
        ).pack(
            side=tk.LEFT,
            padx=20
        )
        self.figure = plt.Figure(
            figsize=(14, 8),
            dpi=100
        )
        self.ax = self.figure.add_subplot(111)
        self.canvas = FigureCanvasTkAgg(
            self.figure,
            master=self.root
        )
        self.canvas_widget = (
            self.canvas.get_tk_widget()
        )
        self.canvas_widget.pack(
            side=tk.TOP,
            fill=tk.BOTH,
            expand=True
        )
        self.toolbar = NavigationToolbar2Tk(
            self.canvas,
            self.root
        )
        self.toolbar.update()
        self.canvas.mpl_connect(
            "scroll_event",
            self.mouse_zoom
        )
        self.program_entry.bind(
            "<Return>",
            lambda event: self.load_and_plot()
        )
        self.order_entry.bind(
            "<Return>",
            lambda event: self.load_and_plot()
        )
        self.ax.set_title(
            "Enter Prog_Nr and Ordername"
        )
        self.ax.set_xlabel(
            "Timestamp"
        )
        self.ax.set_ylabel(
            "OD_real"
        )
        self.ax.grid(
            True,
            linestyle=":",
            alpha=0.5
        )
        self.canvas.draw()
        self.program_entry.focus()
    def get_segment_length(
        self,
        df,
        start_time,
        end_time
    ):
        segment = df[
            (df["timestamp"] >= start_time)
            &
            (df["timestamp"] < end_time)
        ]
        return (
            segment["mandrel_m"]
            .fillna(0)
            .sum()
        )
    def calculate_final_mandrel_meters(
        self,
        df
    ):
        result = (
            df[
                [
                    "mandrel_no",
                    "mandrel_m"
                ]
            ]
            .copy()
        )
        result["mandrel_no"] = pd.to_numeric(
            result["mandrel_no"],
            errors="coerce"
        )
        result["mandrel_m"] = pd.to_numeric(
            result["mandrel_m"],
            errors="coerce"
        )
        result = result.dropna(
            subset=[
                "mandrel_no",
                "mandrel_m"
            ]
        )
        result["mandrel_no"] = (
            result["mandrel_no"]
            .astype(int)
        )
        final_lengths = (
            result
            .groupby(
                "mandrel_no",
                as_index=False
            )["mandrel_m"]
            .sum()
            .rename(
                columns={
                    "mandrel_m":
                    "final_mandrel_m"
                }
            )
            .sort_values(
                "mandrel_no"
            )
            .reset_index(drop=True)
        )
        return final_lengths
    def adjust_mandrel_boundaries(
        self,
        df
    ):
        df = (
            df
            .sort_values("timestamp")
            .reset_index(drop=True)
            .copy()
        )
        boundary_mask = (
            df["mandrel_no"].ne(
                df["mandrel_no"].shift()
            )
        )
        original_boundaries = df.loc[
            boundary_mask
            &
            (df.index > 0),
            [
                "timestamp",
                "mandrel_no"
            ]
        ].copy()
        if original_boundaries.empty:
            return df, []
        df["OD_smooth"] = (
            df["OD_real"]
            .rolling(
                window=5,
                center=True,
                min_periods=1
            )
            .mean()
        )
        df["od_change"] = (
            df["OD_smooth"].diff()
        )
        all_changes = (
            df["od_change"]
            .dropna()
            .to_numpy()
        )
        if len(all_changes) == 0:
            threshold = MIN_OD_DROP
        else:
            median_change = np.median(
                all_changes
            )
            mad = np.median(
                np.abs(
                    all_changes
                    - median_change
                )
            )
            robust_sigma = (
                1.4826 * mad
            )
            threshold = max(
                MIN_OD_DROP,
                DROP_MAD_MULTIPLIER
                * robust_sigma
            )
        all_significant_drops = df[
            df["od_change"] <= -threshold
        ].copy()
        corrected_boundaries = []
        for _, boundary in (
            original_boundaries.iterrows()
        ):
            original_time = (
                boundary["timestamp"]
            )
            window_start = (
                original_time
                -
                pd.Timedelta(
                    minutes=
                    BOUNDARY_SEARCH_MINUTES
                )
            )
            window_end = (
                original_time
                +
                pd.Timedelta(
                    minutes=
                    BOUNDARY_SEARCH_MINUTES
                )
            )
            nearby_drops = (
                all_significant_drops[
                    (
                        all_significant_drops[
                            "timestamp"
                        ] >= window_start
                    )
                    &
                    (
                        all_significant_drops[
                            "timestamp"
                        ] <= window_end
                    )
                ]
            )
            if nearby_drops.empty:
                corrected_boundaries.append(
                    original_time
                )
            else:
                strongest_drop = (
                    nearby_drops.loc[
                        nearby_drops[
                            "od_change"
                        ].idxmin()
                    ]
                )
                corrected_boundaries.append(
                    strongest_drop["timestamp"]
                )
        optimized_boundaries = (
            corrected_boundaries.copy()
        )
        for i in range(
            1,
            len(optimized_boundaries) - 1
        ):
            current_boundary = (
                optimized_boundaries[i]
            )
            search_start = (
                current_boundary
                -
                pd.Timedelta(
                    minutes=
                    BOUNDARY_SEARCH_MINUTES
                )
            )
            search_end = (
                current_boundary
                +
                pd.Timedelta(
                    minutes=
                    BOUNDARY_SEARCH_MINUTES
                )
            )
            candidate_drops = (
                all_significant_drops[
                    (
                        all_significant_drops[
                            "timestamp"
                        ] >= search_start
                    )
                    &
                    (
                        all_significant_drops[
                            "timestamp"
                        ] <= search_end
                    )
                ]
            )
            if candidate_drops.empty:
                continue
            best_boundary = (
                current_boundary
            )
            best_score = float("inf")
            previous_boundary = (
                optimized_boundaries[i - 1]
            )
            next_boundary = (
                optimized_boundaries[i + 1]
            )
            for _, candidate in (
                candidate_drops.iterrows()
            ):
                candidate_time = (
                    candidate["timestamp"]
                )
                length_before = (
                    self.get_segment_length(
                        df,
                        previous_boundary,
                        candidate_time
                    )
                )
                length_after = (
                    self.get_segment_length(
                        df,
                        candidate_time,
                        next_boundary
                    )
                )
                error_before = abs(
                    length_before
                    -
                    MANDREL_LENGTH
                )
                error_after = abs(
                    length_after
                    -
                    MANDREL_LENGTH
                )
                score = (
                    error_before
                    +
                    error_after
                )
                if score < best_score:
                    best_score = score
                    best_boundary = (
                        candidate_time
                    )
            optimized_boundaries[i] = (
                best_boundary
            )
        corrected_boundaries = sorted(
            optimized_boundaries
        )
        first_theoretical_boundary = (
            original_boundaries[
                "timestamp"
            ].min()
        )
        first_window_end = (
            first_theoretical_boundary
            +
            pd.Timedelta(
                minutes=
                BOUNDARY_SEARCH_MINUTES
            )
        )
        first_section = (
            all_significant_drops[
                all_significant_drops[
                    "timestamp"
                ] < first_window_end
            ]
        )
        if not first_section.empty:
            earliest_drop = (
                first_section
                .sort_values("timestamp")
                .iloc[0]
            )
            first_drop_time = (
                earliest_drop["timestamp"]
            )
            if corrected_boundaries:
                nearest_boundary_distance = min(
                    abs(
                        (
                            first_drop_time
                            -
                            b
                        ).total_seconds()
                    )
                    for b in corrected_boundaries
                )
                if (
                    nearest_boundary_distance
                    >
                    BOUNDARY_SEARCH_MINUTES
                    * 60
                ):
                    corrected_boundaries.append(
                        first_drop_time
                    )
        corrected_boundaries = sorted(
            corrected_boundaries
        )
        boundary_count = len(
            corrected_boundaries
        )
        total_mandrels = (
            boundary_count + 1
        )
        df["mandrel_no"] = (
            total_mandrels
        )
        for idx, boundary_time in enumerate(
            corrected_boundaries
        ):
            new_number = (
                total_mandrels
                -
                idx
                -
                1
            )
            df.loc[
                df["timestamp"]
                >= boundary_time,
                "mandrel_no"
            ] = new_number
        df["mandrel_no"] = (
            df["mandrel_no"]
            .clip(lower=1)
            .astype(int)
        )
        df.drop(
            columns=[
                "OD_smooth",
                "od_change"
            ],
            inplace=True,
            errors="ignore"
        )
        return (
            df,
            corrected_boundaries
        )
    def remove_initial_short_mandrel(
        self,
        df
    ):
        check_df = (
            df[
                [
                    "timestamp",
                    "OD_real",
                    "mandrel_no"
                ]
            ]
            .copy()
            .sort_values("timestamp")
            .reset_index(drop=True)
        )
        check_df["OD_real"] = pd.to_numeric(
            check_df["OD_real"],
            errors="coerce"
        )
        check_df = (
            check_df
            .dropna(
                subset=[
                    "timestamp",
                    "OD_real",
                    "mandrel_no"
                ]
            )
            .reset_index(drop=True)
        )
        if check_df.empty:
            return df
        current_mandrel_count = (
            df["mandrel_no"]
            .dropna()
            .astype(int)
            .nunique()
        )
        if (
            current_mandrel_count
            <=
            MIN_MANDREL_COUNT_FOR_REMOVAL
        ):
            return df
        producing_mask = (
            check_df["OD_real"]
            >
            STOP_OD_THRESHOLD
        )
        producing_rows = (
            check_df[
                producing_mask
            ]
        )
        if producing_rows.empty:
            return df
        first_production_index = (
            producing_rows.index[0]
        )
        first_production_time = (
            check_df.loc[
                first_production_index,
                "timestamp"
            ]
        )
        interruption_mask = (
            check_df["OD_real"]
            <=
            STOP_OD_THRESHOLD
        )
        interruption_rows = (
            check_df.loc[
                (
                    check_df.index
                    >
                    first_production_index
                )
                &
                interruption_mask
            ]
        )
        if interruption_rows.empty:
            return df
        first_interruption_index = (
            interruption_rows.index[0]
        )
        first_interruption_time = (
            check_df.loc[
                first_interruption_index,
                "timestamp"
            ]
        )
        interruption_seconds = (
            first_interruption_time
            -
            first_production_time
        ).total_seconds()
        if (
            interruption_seconds
            >=
            INITIAL_INTERRUPTION_MAX_SECONDS
        ):
            return df
        highest_mandrel = int(
            df["mandrel_no"].max()
        )
        if highest_mandrel <= 1:
            return df
        rows_to_remove = (
            df["mandrel_no"]
            ==
            highest_mandrel
        )
        df.loc[
            rows_to_remove,
            "mandrel_no"
        ] = (
            highest_mandrel - 1
        )
        df["mandrel_no"] = (
            df["mandrel_no"]
            .clip(lower=1)
            .astype(int)
        )
        return df
    def load_and_plot(self):
        program = (
            self.program_var
            .get()
            .strip()
        )
        order = (
            self.order_var
            .get()
            .strip()
        )
        if not program:
            messagebox.showwarning(
                "Missing Prog_Nr",
                "Please enter a Prog_Nr."
            )
            self.program_entry.focus()
            return
        if not order:
            messagebox.showwarning(
                "Missing Ordername",
                "Please enter an Ordername."
            )
            self.order_entry.focus()
            return
        self.status_var.set(
            "Loading selected order from SQL Server..."
        )
        self.load_button.config(
            state="disabled"
        )
        self.root.update_idletasks()
        conn = None
        try:
            conn = pyodbc.connect(
                CONNECTION_STRING
            )
            chunks = pd.read_sql(
                query,
                conn,
                params=[
                    START_DATE,
                    program,
                    order
                ],
                chunksize=50000
            )
            df = pd.concat(
                chunks,
                ignore_index=True
            )
            if df.empty:
                messagebox.showinfo(
                    "No data",
                    f"No data found for:\n\n"
                    f"Prog_Nr: {program}\n"
                    f"Ordername: {order}\n\n"
                    f"Start date: {START_DATE}"
                )
                self.status_var.set(
                    "No data found."
                )
                return
            df["timestamp"] = (
                pd.to_datetime(
                    df["timestamp"],
                    errors="coerce"
                )
            )
            df["ID"] = (
                df["ID"]
                .astype(str)
                .str.replace(
                    ",",
                    ".",
                    regex=False
                )
            )
            df["ID"] = pd.to_numeric(
                df["ID"],
                errors="coerce"
            )
            df["wth"] = (
                df["wth"]
                .astype(str)
                .str.replace(
                    ",",
                    ".",
                    regex=False
                )
            )
            df["wth"] = pd.to_numeric(
                df["wth"],
                errors="coerce"
            )
            df = (
                df
                .sort_values("timestamp")
                .reset_index(drop=True)
            )
            df["mandrel_m"] = pd.to_numeric(
                df["mandrel_m"],
                errors="coerce"
            )
            df["mandrel_m"] = (
                df["mandrel_m"]
                .replace(
                    [
                        np.inf,
                        -np.inf
                    ],
                    np.nan
                )
            )
            df.loc[
                df["mandrel_m"] < 0,
                "mandrel_m"
            ] = np.nan
            df["produced_length_reverse"] = (
                df["mandrel_m"]
                .fillna(0)
                .iloc[::-1]
                .cumsum()
                .iloc[::-1]
                .values
            )
            df["mandrel_no"] = (
                np.floor(
                    df["produced_length_reverse"]
                    /
                    MANDREL_LENGTH
                )
                +
                1
            ).astype(int)
            boundary_mask = (
                (
                    df["produced_length_reverse"]
                    >
                    0
                )
                &
                (
                    (
                        df["produced_length_reverse"]
                        %
                        MANDREL_LENGTH
                    )
                    .round(5)
                    ==
                    0
                )
            )
            df.loc[
                boundary_mask,
                "mandrel_no"
            ] -= 1
            df["mandrel_no"] = (
                df["mandrel_no"]
                .clip(lower=1)
                .astype(int)
            )
            df.drop(
                columns=[
                    "produced_length_reverse"
                ],
                inplace=True
            )
            df, corrected_boundaries = (
                self.adjust_mandrel_boundaries(
                    df
                )
            )
            df = (
                self.remove_initial_short_mandrel(
                    df
                )
            )
            final_mandrel_lengths = (
                self.calculate_final_mandrel_meters(
                    df
                )
            )
            df = df.merge(
                final_mandrel_lengths,
                on="mandrel_no",
                how="left"
            )
            df["total_m_order"] = (
                df["mandrel_no"]
                *
                MANDREL_LENGTH
            ).round(2)
            df.rename(
                columns={
                    "Linie_ist": "line_speed"
                },
                inplace=True
            )
            df["compound"] = (
                df["compound"]
                .fillna("")
                .astype(str)
                .str.strip()
            )
            df["Prog_Nr"] = (
                df["Prog_Nr"]
                .fillna("")
                .astype(str)
                .str.strip()
            )
            df["ordername"] = (
                df["ordername"]
                .fillna("")
                .astype(str)
                .str.strip()
            )
            numeric_cols = [
                "ID",
                "wth",
                "OD_real",
                "ID_real",
                "wt_real",
                "line_speed",
                "rpm",
                "mandrel_no",
                "mandrel_m",
                "final_mandrel_m",
                "total_m_order",
                "mass_pressure",
                "mass_temp"
            ]
            for col in numeric_cols:
                df[col] = pd.to_numeric(
                    df[col],
                    errors="coerce"
                )
                df[col] = (
                    df[col]
                    .replace(
                        [
                            np.inf,
                            -np.inf
                        ],
                        np.nan
                    )
                )
            df = df.dropna(
                subset=[
                    "timestamp",
                    "OD_real",
                    "mandrel_no"
                ]
            )
            if df.empty:
                messagebox.showinfo(
                    "No valid data",
                    "The selected order exists, "
                    "but there is no valid data for "
                    "timestamp / OD_real / mandrel."
                )
                self.status_var.set(
                    "No valid graph data."
                )
                return
            self.plot_data(
                df,
                program,
                order
            )
        except Exception as e:
            messagebox.showerror(
                "Database / Processing Error",
                str(e)
            )
            self.status_var.set(
                "Error loading data."
            )
        finally:
            if conn is not None:
                try:
                    conn.close()
                except Exception:
                    pass
            self.load_button.config(
                state="normal"
            )
    def plot_data(
        self,
        plot_df,
        program,
        order
    ):
        self.ax.clear()
        plot_df = (
            plot_df
            .sort_values("timestamp")
            .reset_index(drop=True)
        )
        od_line = self.ax.plot(
            plot_df["timestamp"],
            plot_df["OD_real"],
            color="blue",
            linewidth=0.8,
            label="OD_real",
            zorder=3
        )[0]
        self.ax.set_ylabel(
            "OD_real",
            fontsize=11,
            color="blue"
        )
        self.ax.tick_params(
            axis="y",
            labelcolor="blue"
        )
        mandrel_change = (
            plot_df["mandrel_no"].ne(
                plot_df["mandrel_no"].shift()
            )
        )
        change_rows = plot_df[
            mandrel_change
            &
            (plot_df.index > 0)
        ]
        for _, row in change_rows.iterrows():
            timestamp = (
                row["timestamp"]
            )
            mandrel_no = int(
                row["mandrel_no"]
            )
            self.ax.axvline(
                x=timestamp,
                color="red",
                linestyle="--",
                linewidth=1.3,
                alpha=0.85,
                zorder=1
            )
            self.ax.text(
                timestamp,
                0.98,
                f"M{mandrel_no}",
                transform=self.ax.get_xaxis_transform(),
                rotation=90,
                color="red",
                fontsize=8,
                fontweight="bold",
                verticalalignment="top",
                horizontalalignment="right"
            )
        self.ax.set_xlabel(
            "Timestamp",
            fontsize=11
        )
        self.ax.set_title(
            f"OD_real vs Timestamp\n"
            f"Prog_Nr: {program} | "
            f"Ordername: {order}\n"
            f"Mandrel calculation: END → START",
            fontsize=13,
            fontweight="bold"
        )
        self.ax.grid(
            True,
            which="major",
            linestyle=":",
            alpha=0.5
        )
        self.ax.xaxis.set_major_formatter(
            mdates.DateFormatter(
                "%Y-%m-%d\n%H:%M:%S"
            )
        )
        self.figure.autofmt_xdate()
        self.ax.legend(
            [od_line],
            ["OD_real"],
            loc="upper right"
        )
        mandrel_numbers = (
            plot_df["mandrel_no"]
            .dropna()
            .astype(int)
            .unique()
        )
        mandrel_count = len(
            mandrel_numbers
        )
        start_mandrel = int(
            plot_df.iloc[0]["mandrel_no"]
        )
        end_mandrel = int(
            plot_df.iloc[-1]["mandrel_no"]
        )
        total_length = (
            mandrel_count
            *
            MANDREL_LENGTH
        )
        self.status_var.set(
            f"Prog_Nr: {program} | "
            f"Order: {order} | "
            f"Rows: {len(plot_df):,} | "
            f"Mandrels: {mandrel_count:,} | "
            f"START: M{start_mandrel} | "
            f"END: M{end_mandrel} | "
            f"Length: {total_length:,.1f} m"
        )
        final_lengths = (
            plot_df[
                [
                    "mandrel_no",
                    "final_mandrel_m"
                ]
            ]
            .drop_duplicates()
            .sort_values("mandrel_no")
        )
        if not final_lengths.empty:
            length_lines = []
            for _, row in (
                final_lengths.iterrows()
            ):
                mandrel_no = int(
                    row["mandrel_no"]
                )
                meters = float(
                    row["final_mandrel_m"]
                )
                length_lines.append(
                    f"M{mandrel_no} = "
                    f"{meters:.2f} m"
                )
            information_text = (
                "FINAL MANDrEL LENGTHS\n"
                +
                "\n".join(length_lines)
            )
            self.ax.text(
                0.01,
                0.98,
                information_text,
                transform=self.ax.transAxes,
                fontsize=8,
                verticalalignment="top",
                horizontalalignment="left",
                bbox=dict(
                    boxstyle="round",
                    facecolor="white",
                    edgecolor="gray",
                    alpha=0.85
                )
            )
        self.figure.tight_layout()
        self.canvas.draw()
    def clear_graph(self):
        self.ax.clear()
        self.ax.set_title(
            "Enter Prog_Nr and Ordername"
        )
        self.ax.set_xlabel(
            "Timestamp"
        )
        self.ax.set_ylabel(
            "OD_real"
        )
        self.ax.grid(
            True,
            linestyle=":",
            alpha=0.5
        )
        self.status_var.set(
            "Enter Prog_Nr and Ordername"
        )
        self.canvas.draw()
    def mouse_zoom(
        self,
        event
    ):
        if event.inaxes != self.ax:
            return
        if event.button == "up":
            scale_factor = 0.7
        elif event.button == "down":
            scale_factor = 1.4
        else:
            return
        x_min, x_max = (
            self.ax.get_xlim()
        )
        y_min, y_max = (
            self.ax.get_ylim()
        )
        x_mouse = event.xdata
        y_mouse = event.ydata
        if x_mouse is None:
            return
        if y_mouse is None:
            return
        new_x_min = (
            x_mouse
            -
            (
                x_mouse
                -
                x_min
            )
            *
            scale_factor
        )
        new_x_max = (
            x_mouse
            +
            (
                x_max
                -
                x_mouse
            )
            *
            scale_factor
        )
        new_y_min = (
            y_mouse
            -
            (
                y_mouse
                -
                y_min
            )
            *
            scale_factor
        )
        new_y_max = (
            y_mouse
            +
            (
                y_max
                -
                y_mouse
            )
            *
            scale_factor
        )
        self.ax.set_xlim(
            new_x_min,
            new_x_max
        )
        self.ax.set_ylim(
            new_y_min,
            new_y_max
        )
        self.canvas.draw_idle()
if __name__ == "__main__":
    root = tk.Tk()
    app = MandrelVisualizer(
        root
    )
    root.mainloop()


C:\Users\velagaturi\AppData\Local\Temp\ipykernel_20252\3143778337.py:772: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  chunks = pd.read_sql(
